# 15.4 World Models — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter15_4_world_models.ipynb)

책 본문: [15.4 World Models](https://smhanlab.com/book-ml/kor/ml2/chapter15/4.html)

작은 합성 환경(또는 CartPole)에서 다음 상태를 예측하는 작은 신경망 다이내믹스 모델을 학습하고, 그 학습된 모델 '안에서' 몇 스텝 상상 롤아웃을 해보는 감을 잡습니다.


## 개요: World Models의 세 단계

책 15.4절의 아이디어를 최소 규모로 실행한다 — (1) 무작위 액션으로 모은
CartPole 데이터로 **작은 신경망 다이내믹스 모델**을 학습하고, (2) 이 모델
**안에서만** 정책을 롤아웃(imagined rollout)하고, (3) 상상 궤적이 실제와
얼마나 비슷한지 확인한다.

핵심: 이 노트북에서 실제 환경(`env.step`)은 데이터 수집(1절)과 마지막
검증(3~4절)에만 쓰인다 — 그 사이에서 정책이 "살아보는" 곳은 전부
학습된 모델, 즉 **상상**이다.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):
    IMG = "/tmp"

import gymnasium as gym
import torch
import torch.nn as nn

torch.manual_seed(0)
rng = np.random.default_rng(0)

print("torch:", torch.__version__, " gymnasium:", gym.__version__)
print("한글 폰트:", kr[0] if kr else "NOT FOUND — 한글 표시가 깨질 수 있음")

torch: 2.13.0+cpu  gymnasium: 1.3.0
한글 폰트: Noto Sans CJK KR


## 1. 데이터: 무작위 액션으로 CartPole 경험 모으기

World Model의 출발은 "환경이 어떻게 움직이는지"라는 데이터다. 정책이
별로인(무작위) 액션을 400에피소드간 쏟아붓고,
\\((s_t, a_t, s_{t+1})\\) 튜플을 모은다. 이 단계만 실제 환경이 쓰인다 —
이후 정책이 상상 속 경험을 만드는 데는 모델이 이 경험을 "기억한"
규칙을 쓴다.

In [2]:
# CartPole-v1: 상태 (x, x', θ, θ') — 카트 위치·속도, 막대 각도·각속도.
# 액션 0 = 왼쪽, 1 = 오른쪽. θ가 ±12°(0.2095 rad)를 벗어나면 떨어짐.
env = gym.make("CartPole-v1")
THETA_MAX = 0.2095
H = 100  # 상상 롤아웃 길이(스텝)

samples = []
for ep in range(400):
    obs, _ = env.reset(seed=ep)
    for t in range(500):
        a = int(rng.integers(2))          # 무작위 액션(탐험)
        nxt, r, term, trunc, _ = env.step(a)
        samples.append((obs, a, nxt))     # (s_t, a_t, s_{t+1})
        obs = nxt
        if term or trunc:
            break

# (s_t, a_t) → s_{t+1} 학습 데이터. 입력은 [현재 상태 4차원, 액션],
# 출력은 다음 상태 전체 4차원이다. 값이 크지 않게 정규화해 둔다.
X = np.zeros((len(samples), 5), dtype=np.float32)
Y = np.zeros((len(samples), 4), dtype=np.float32)
for i, (o, a, no) in enumerate(samples):
    X[i, 0:4] = o
    X[i, 4] = a
    Y[i] = no
xstd = X.std(axis=0); xstd[xstd == 0] = 1.0
ystd = Y.std(axis=0); ystd[ystd == 0] = 1.0

print(f"모은 (s, a, s') 샘플 수: {len(samples):,}")
print("입력 차원:", X.shape[1], "→ 출력(다음 상태) 차원:", Y.shape[1])

모은 (s, a, s') 샘플 수: 9,297
입력 차원: 5 → 출력(다음 상태) 차원: 4


## 2. World Model: "다음 상태"를 예측하는 작은 신경망

다이내믹스 모델은 단순한 회귀 문제다 — \\((s\_t, a\_t) \to s\_{t+1}\\)을
입력으로 받아 다음 상태 전체를 출력하도록 작은 MLP를 학습한다. 9,000여
샘플로 80에포크 학습하면 1스텝 예측 오차가 각도 기준 약 0.001 rad
(≈0.08°) 수준으로 줄어든다 — CartPole이 디테일한 물리 시뮬레이터임에도
"다음 1스텝"은 거의 완벽히 예측할 수 있게 된다.

In [3]:
Xn = torch.tensor(X / xstd)
Yn = torch.tensor(Y / ystd)

# 5 → 96 → 96 → 4 MLP. (s, a) 를 받아 다음 상태 전체를 예측한다.
model = nn.Sequential(
    nn.Linear(5, 96), nn.ReLU(),
    nn.Linear(96, 96), nn.ReLU(),
    nn.Linear(96, 4),
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

for epoch in range(80):
    perm = rng.permutation(len(Xn))
    for i in range(0, len(Xn), 1000):
        idx = torch.tensor(perm[i:i + 1000])
        loss = loss_fn(model(Xn[idx]), Yn[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 20 == 0 or epoch == 79:
        with torch.no_grad():
            err = (model(Xn).numpy() * ystd - Y) ** 2
            rmse = np.sqrt(err.mean(axis=0))
        print(f"epoch {epoch:2d}  1스텝 예측 RMSE: x={rmse[0]:.4f}  v={rmse[1]:.4f}  "
              f"θ={rmse[2]:.4f} rad  ω={rmse[3]:.4f}")

epoch  0  1스텝 예측 RMSE: x=0.0605  v=0.3859  θ=0.0674 rad  ω=0.6015


epoch 20  1스텝 예측 RMSE: x=0.0020  v=0.0158  θ=0.0026 rad  ω=0.0282


epoch 40  1스텝 예측 RMSE: x=0.0013  v=0.0102  θ=0.0018 rad  ω=0.0165


epoch 60  1스텝 예측 RMSE: x=0.0011  v=0.0086  θ=0.0015 rad  ω=0.0138


epoch 79  1스텝 예측 RMSE: x=0.0010  v=0.0075  θ=0.0013 rad  ω=0.0125


## 3. 상상 롤아웃: 모델 안에서의 정책

이제 **실제 환경에 묻지 않고** 학습된 모델 안에서만 정책을 굴려본다.
`imagine` 함수는 (1) 정책을 현재 상태에 적용해 액션을 뽑고, (2) **모델의
순전파**로 다음 상태를 만들고, (3) 이를 반복한다 — 이 루프의 어디에도
`env.step`이 없다.

네 가지 정책을 비교한다:

- `always right` / `always left`: 액션을 고정 (1 / 0) — 막대는 한쪽으로만 기울어져 빠르게 떨어진다.
- `good linear`: `a = 1 if [2, 1, 20, 5]·s ≥ 0 else 0` — 간단한 피드백 제어기로, 막대를 충분히 오래 세운다.
- `sign-flipped`: 위 제어기의 부호만 뒤집은 `a = 1 if [0, 0, -20, -5]·s ≥ 0 else 0` — 오히려 막대를 **더 빨리** 떨어뜨린다.

좋은 정책이든 나쁜 정책이든, 정책이 "좋냐 나쁘냐"는 판단은 **상상
궤적이 얼마나 오래 지속되는지**로 모델 스스로 판정한다.

In [4]:
def make_input(o, a):
    """(s_t, a_t) → 모델 입력 (정규화)."""
    x = np.zeros(5, dtype=np.float32)
    x[0:4] = o
    x[4] = a
    return torch.tensor(x / xstd)

def step_model(o, a):
    """모델의 순전파 한 스텝 — 실제 환경을 쓰지 않는다."""
    with torch.no_grad():
        return np.asarray(model(make_input(o, a)).numpy() * ystd, dtype=np.float32)

def imagine(policy, start, H=H):
    """정책을 학습된 모델 안에서만 롤아웃. 막대(θ)가 THETA_MAX를
    벗어나면 에피소드 종료(상상 속 '떨어짐'). 반환: (θ 궤적, 생존 스텝)."""
    o = start.copy()
    traj, survival = [], H
    for t in range(H):
        a = int(policy(o))
        o = step_model(o, a)            # ← 실제 환경이 아닌 모델이 다음 상태를 만든다
        traj.append(o[2])
        if abs(o[2]) >= THETA_MAX:
            survival = t
            break
    return np.array(traj), survival

# 실제 환경과의 정직한 비교를 위해, 상상도 실제 reset과 같은 초기 상태에서
s0 = np.asarray(env.reset(seed=123)[0], dtype=np.float32)

controllers = {
    "always right": (np.zeros(4), 1.0),
    "always left":  (np.zeros(4), 0.0),
    "good linear":  (np.array([2.0, 1.0, 20.0, 5.0]), None),
    "sign-flipped": (np.array([0.0, 0.0, -20.0, -5.0]), None),
}

def make_policy(K, const):
    if const is not None:
        return lambda o: const
    return lambda o: 1.0 if float(K @ o) >= 0 else 0.0

def real_rollout(policy, H=H, seed=123):
    """같은 정책을 실제 gymnasium 환경에서 롤아웃(imagine의 대조군)."""
    o, _ = env.reset(seed=seed)
    traj, survival = [], H
    for t in range(H):
        a = int(policy(o))
        o, r, term, trunc, _ = env.step(a)
        traj.append(o[2])
        if abs(o[2]) >= THETA_MAX or term or trunc:
            survival = t
            break
    return np.array(traj), survival

print("모든 롤아웃의 초기 상태 (실제 reset, seed=123):", np.round(s0, 3))
print()
print(f"{'policy':15s} {'imagine 생존':>12s} {'real 생존':>10s}")
for name, (K, const) in controllers.items():
    p = make_policy(K, const)
    _, i_surv = imagine(p, s0)
    _, r_surv = real_rollout(p)
    print(f"{name:15s} {i_surv:12d} {r_surv:10d}")

모든 롤아웃의 초기 상태 (실제 reset, seed=123): [ 0.018 -0.045 -0.028 -0.032]

policy            imagine 생존    real 생존
always right               8          8
always left               10          9
good linear              100        100
sign-flipped               8          8


## 4. 상상이 얼마나 "진짜"인가: 궤적 비교와 그림

생존 스텝이 일치했더라도 **궤적의 세부**는 다를 수 있다. `good linear`
정책으로 실제로 롤아웃한 궤적과 상상한 궤적을 겹쳐 본다. 1스텝 예측
오차(각도 기준 ≈0.001 rad)는 스텝을 이어 붙일수록 조금씩 쌓인다 —
여기서는 20스텝쯤에 ≈0.02 rad(≈1°)로 벌어졌다가, 제어기가 막대를
정중앙으로 되돌리며 다시 좁혀진다. **생존 판정(떨어지느냐 마느냐)에는
영향을 주지 않을 만큼 작다** — 이 작은 환경에서 100스텝 상상이
"충분히 진짜"인 이유다.

반면 모델이 정확하지 않거나(더 적은 데이터로 학습) 세계가 더 불안정
하면 이 오차는 **스텝마다 복리로 쌓여** 긴 롤아웃에서 판정을 완전히
틀리게 만든다 — 세계가 얼마나 불안정한지, 모델이 얼마나 정확한지는
World Model의 "상상"을 얼마나 신뢰할 수 있는지 직접 결정한다.
Dreamer 계열은 이 한계를 (a) 압축된 잠재 공간에서 상상하고, (b)
상상 길이(horizon)를 제한하고, (c) 모델 오차에 강건한 정책·가치
함수를 학습하는 쪽으로 공략한다.

In [5]:
# (a) 네 정책의 상상 vs 실제 생존 스텝  (b) good linear의 실제/상상 θ 궤적
p_good = make_policy(*controllers["good linear"])
good_imag_traj, _ = imagine(p_good, s0)
good_real_traj, _ = real_rollout(p_good)

surv_names = list(controllers.keys())
surv_imag, surv_real = [], []
for name in surv_names:
    p = make_policy(*controllers[name])
    _, si = imagine(p, s0)
    _, sr = real_rollout(p)
    surv_imag.append(si)
    surv_real.append(sr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
xx = np.arange(len(surv_names))
w = 0.38
ax.bar(xx - w / 2, surv_imag, width=w, label="imagine (학습된 모델)", color="#4c72b0")
ax.bar(xx + w / 2, surv_real, width=w, label="real (실제 환경)", color="#dd8452")
for i, (si, sr) in enumerate(zip(surv_imag, surv_real)):
    ax.text(i - w / 2, si + 2, str(si), ha="center", fontsize=9)
    ax.text(i + w / 2, sr + 2, str(sr), ha="center", fontsize=9)
ax.axhline(H, color="gray", ls=":", lw=1)
ax.text(3.45, H - 6, "H (100)", fontsize=8, color="gray")
ax.set_xticks(xx, surv_names, rotation=18)
ax.set_ylim(0, 115)
ax.set_ylabel("생존 스텝 (떨어지기까지)")
ax.set_title("(a) 생존 스텝: 상상 vs 실제 — 정책 구별은 정확히 일치")
ax.legend(loc="upper right", fontsize=9)

ax = axes[1]
ax.plot(good_real_traj, label="실제 환경", lw=1.6, color="#1f77b4")
ax.plot(good_imag_traj, label="학습된 모델 (상상)", lw=1.6, color="#ff7f0e")
ax.axhline(THETA_MAX, color="red", ls="--", lw=0.8)
ax.axhline(-THETA_MAX, color="red", ls="--", lw=0.8)
ax.text(2, THETA_MAX + 0.004, "떨어짐 임계 ±12°", color="red", fontsize=8)
ax.set_xlabel("스텝")
ax.set_ylabel("막대 각도 θ (rad)")
ax.set_title("(b) good linear: 실제 vs 상상 궤적 — 약 0.02 rad(≈1°) 이내로 동행")
ax.legend(fontsize=9)

fig.tight_layout()
p = os.path.join(IMG, "ch15_4_world_models.svg")
fig.savefig(p, bbox_inches="tight")
plt.show()
print("그림 저장:", p)

그림 저장: /home/smhan/book-ml/kor/src/images/ch15_4_world_models.svg


## 5. 정리: 이 데모가 보여준 것

1. **다이내믹스 모델은 "다음 상태 회귀"일 뿐이다** — (s, a) → s'을
   MLP로 배우면 1스텝 예측 오차가 각도 기준 ≈0.001 rad까지 줄었다.
2. **모델 안에서의 롤아웃은 실제 환경을 쓰지 않는다** — `imagine`의
   루프는 전부 모델의 순전파. 이 "상상"으로 네 정책을 구분
   (좋은 정책: 100스텝 생존, 나쁜 정책: 8~10스텝)할 수 있었고, 그
   구별은 실제 환경의 결과(100 vs 8~9)와 일치했다.
3. **상상은 1스텝 오차를 스텝마다 쌓는다** — good linear 궤적은 20스텝쯤에
   ≈0.02 rad(≈1°) 차이로 벌어졌지만, 생존 판정에는 영향이 없었다.
   모델의 정확도와 세계의 불안정성이 "상상을 얼마나 신뢰할 수 있는가"
   를 정한다는 것이 World Model 방법론 전체의 핵심 긴장이다.

이 데모는 책 15.4절의 **World Models**(Ha & Schmidhuber, 2018)의
"latent 공간에서만 정책 학습"과 **Dreamer**(Hafner et al., 2020)의
"상상 궤적으로 역전파해 정책 학습"의 **개념**만 재현했다 — 잠재 공간
(VAE)나 상상 역전파는 생략하고, "학습된 모델 안에서 정책이
경험을 만든다"는 한 가지 아이디어만 가장 작은 규모로 보여줬다.